# Experiment 16 — Theme Business Needs — Theme Batch, Custom Prompt, IDs Only

## What the LLM sees

One call per Theme for the sampled valid Epics in that Theme. Shared Theme Business Needs; each Epic carries only its own Stage(s) and candidate L3s. Theme Description and ground truth are never sent.

## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import json
import os

import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
)

NOTEBOOK_DIR = Path.cwd()


def resolve_data_path(relative_path: str, env_var: str) -> Path:
    """Resolve a data file from an override, notebook folder, or repo root."""
    override = os.getenv(env_var)
    if override:
        return Path(override).expanduser()

    relative_path = Path(relative_path)
    search_roots = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR.parent,
        NOTEBOOK_DIR / "l3_experiments",
    ]
    seen = set()
    attempted = []

    for root in search_roots:
        candidate = root / relative_path
        key = str(candidate.resolve(strict=False))
        if key in seen:
            continue
        seen.add(key)
        attempted.append(candidate)
        if candidate.exists():
            return candidate

    tried = "\n  - ".join(str(path) for path in attempted)
    raise FileNotFoundError(
        f"Could not find {relative_path}. Tried:\n  - {tried}"
    )


PARQUET_PATH = resolve_data_path("full_golden.parquet", "L3_FULL_GOLDEN_PATH")
STAGE_PATH = resolve_data_path("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve_data_path(
    "VSSCaprv (1).csv",
    "L3_STAGE_CAPABILITY_MAP_PATH",
)
GROUND_TRUTH_PATH = resolve_data_path(
    "results/epic_l3_ground_truth_full_golden.xlsx",
    "L3_GROUND_TRUTH_PATH",
)

SAMPLE_SIZE = 50
SAMPLE_SEED = 42

# Optional single-example inspection. Leave None for batch execution only.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E16_BUSINESS_NEEDS_THEME_BATCH_CUSTOM_PROMPT"


## Retrieval

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def parse_list_value(value) -> list[str]:
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        return [clean_text(item) for item in value if clean_text(item)]

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]

    if isinstance(parsed, (list, tuple, set)):
        return [clean_text(item) for item in parsed if clean_text(item)]
    return [clean_text(parsed)] if clean_text(parsed) else []


def read_table(path, *, sheet_name=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(
            path,
            dtype=str,
            encoding="cp1252",
            encoding_errors="replace",
        )
    return pd.read_excel(path, sheet_name=sheet_name, dtype=str)


def load_evaluation_population():
    population = read_table(
        GROUND_TRUTH_PATH,
        sheet_name="evaluation_population",
    )
    required = {
        "theme_key",
        "epic_key",
        "stage_ids",
        "gt_l3_ids",
        "candidate_l3_ids",
    }
    missing = required.difference(population.columns)
    if missing:
        raise KeyError(
            f"evaluation_population is missing columns: {sorted(missing)}"
        )

    population = (
        population
        .drop_duplicates(subset=["theme_key", "epic_key"], keep="first")
        .sort_values(["theme_key", "epic_key"], kind="stable")
        .reset_index(drop=True)
    )

    if population["epic_key"].duplicated().any():
        raise ValueError("evaluation_population contains duplicate Epic keys.")

    if len(population) < SAMPLE_SIZE:
        raise ValueError(
            f"Need {SAMPLE_SIZE} valid Epics, found only {len(population)}."
        )

    sample = population.sample(
        n=SAMPLE_SIZE,
        random_state=SAMPLE_SEED,
        replace=False,
    )
    return sample.sort_values(
        ["theme_key", "epic_key"],
        kind="stable",
    ).reset_index(drop=True)


evaluation_population = load_evaluation_population()
selected_pairs = set(
    zip(
        evaluation_population["theme_key"],
        evaluation_population["epic_key"],
    )
)
selected_theme_ids = set(evaluation_population["theme_key"])


def load_themes():
    frame = read_table(PARQUET_PATH)
    required = {"key", "description", "businessNeeds", "epic_keys"}
    missing = required.difference(frame.columns)
    if missing:
        raise KeyError(
            f"full_golden.parquet is missing columns: {sorted(missing)}"
        )

    themes = {}
    found_pairs = set()

    for _, row in frame.iterrows():
        theme_id = clean_text(row.get("key"))
        if theme_id not in selected_theme_ids:
            continue

        selected_epics = []
        for epic_key in parse_list_value(row.get("epic_keys")):
            if (theme_id, epic_key) in selected_pairs:
                selected_epics.append({"key": epic_key})
                found_pairs.add((theme_id, epic_key))

        if not selected_epics:
            continue

        themes[theme_id] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "epics": selected_epics,
        }

    missing_pairs = selected_pairs - found_pairs
    if missing_pairs:
        raise ValueError(
            "Selected Theme/Epic pairs were not found in full_golden.parquet: "
            f"{sorted(missing_pairs)}"
        )

    return themes


themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)


def stage_context(stage_id):
    match = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id
    ]
    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")
    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }


print(
    f"Selected {len(evaluation_population)} valid Epics "
    f"with seed={SAMPLE_SEED} across {len(themes)} Themes."
)
display(evaluation_population.head(50))


## Candidate construction

In [ ]:
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()
    rows = (
        rows.drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = '''\
You are performing Level 3 business capability classification for multiple Epics belonging to one Theme.

An L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.

The Theme Business Needs is shared context for all Epics in this request.

Each Epic must be classified independently using only:
- the shared Theme Business Needs,
- that Epic's own Value Stream Stage context,
- that Epic's own candidate L3 capabilities.

EVIDENCE

Theme Business Needs is the primary business evidence and describes the business outcomes and needs the Theme is intended to address.

Each Epic's Value Stream Stage defines the business activity boundary relevant to that Epic.

The Stage constrains the interpretation and candidate space. Stage membership alone is not evidence that a candidate capability should be selected.

For each candidate L3:
- capability_id is the exact identifier to return when selected.
- capability_description is the primary semantic definition of the business function.
- capability_name is the supporting business label.
- capability_tier is supporting taxonomy context only.

Do not infer meaning from capability_id.

CLASSIFICATION RULES

For each Epic independently:

1. Determine the business functions supported by Theme Business Needs.
2. Constrain that interpretation using that Epic's Value Stream Stage.
3. Compare the evidence against only that Epic's candidate L3 definitions.
4. Select every candidate whose business function is directly supported.

Do not select a capability merely because:
- it belongs to the Epic's Stage,
- it shares terminology,
- it is broadly related to the Theme,
- it is adjacent, upstream, or downstream,
- it supports another capability.

EPIC ISOLATION

Do not use one Epic's Stage or candidates as evidence for another Epic.

Shared Theme membership does not mean different Epics should receive the same capability selections.

Only return capability_id values supplied under that specific Epic.

If no candidate is supported for an Epic, return an empty list.

Return exactly one result for every supplied epic_key.

OUTPUT

Return JSON only:

{"epics":[{"epic_key":"GROUP-12345","l3":["CAP00000123"]}]}

Do not return reasons, explanations, Markdown, or additional fields.'''

def build_theme_batch_prompt(theme, epic_payloads):
    payload = {
        "task": "Classify every supplied Epic independently using only its own Stage and candidates plus the shared Theme evidence.",
        "theme": {"business_needs": theme["theme_business_needs"]},
        "epics": epic_payloads,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction

In [ ]:
def validate_theme_batch_response(payload, expected_epic_keys, allowed_ids_by_epic):
    if not isinstance(payload, dict) or set(payload) != {"epics"}:
        raise ValueError("Theme-batch response must contain only the epics field.")
    if not isinstance(payload["epics"], list):
        raise ValueError("Theme-batch epics must be a list.")

    expected = list(expected_epic_keys)
    expected_set = set(expected)
    predictions = {}

    for item in payload["epics"]:
        if not isinstance(item, dict) or set(item) != {"epic_key", "l3"}:
            raise ValueError("Each Epic result must contain exactly epic_key and l3.")
        epic_key = str(item["epic_key"]).strip()
        if epic_key not in expected_set:
            raise ValueError(f"Unexpected epic_key: {epic_key}")
        if epic_key in predictions:
            raise ValueError(f"Duplicate epic_key: {epic_key}")
        if not isinstance(item["l3"], list):
            raise ValueError(f"l3 must be a list for {epic_key}")

        allowed = set(allowed_ids_by_epic[epic_key])
        selected = []
        seen = set()
        for capability_id in item["l3"]:
            if not isinstance(capability_id, str):
                raise ValueError(f"L3 IDs must be strings for {epic_key}")
            capability_id = capability_id.strip()
            if capability_id not in allowed:
                raise ValueError(f"{capability_id} is not a candidate for {epic_key}")
            if capability_id in seen:
                raise ValueError(f"Duplicate L3 ID {capability_id} for {epic_key}")
            seen.add(capability_id)
            selected.append(capability_id)
        predictions[epic_key] = selected

    if set(predictions) != expected_set:
        missing = sorted(expected_set - set(predictions))
        raise ValueError(f"Missing Epic results: {missing}")
    return predictions


def build_batch_epics(theme_rows):
    epic_payloads = []
    allowed_ids_by_epic = {}

    for row in theme_rows.to_dict(orient="records"):
        epic_key = str(row["epic_key"]).strip()
        stages = []
        allowed_ids = set()
        for stage_id in parse_list_value(row["stage_ids"]):
            candidates = candidate_rows_for_stage(stage_id)
            stages.append({
                "value_stream_stage": stage_context(stage_id),
                "candidate_l3_capabilities": candidates,
            })
            allowed_ids.update(candidate["capability_id"] for candidate in candidates)
        epic_payloads.append({"epic_key": epic_key, "stages": stages})
        allowed_ids_by_epic[epic_key] = sorted(allowed_ids)

    return epic_payloads, allowed_ids_by_epic


def predict_theme_batch(gateway, theme_id, theme_rows):
    theme = themes[theme_id]
    epic_payloads, allowed_ids_by_epic = build_batch_epics(theme_rows)
    user_prompt = build_theme_batch_prompt(theme, epic_payloads)
    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
        reasoning_effort="low",
    )
    predictions = validate_theme_batch_response(
        parse_json_response(raw_response),
        [item["epic_key"] for item in epic_payloads],
        allowed_ids_by_epic,
    )
    return predictions, metrics, user_prompt, raw_response


## Evaluation

In [ ]:
def run_experiment():
    gateway = load_gateway()
    result_rows = []
    call_rows = []

    for theme_id, theme_rows in evaluation_population.groupby("theme_key", sort=True):
        theme_rows = theme_rows.sort_values("epic_key", kind="stable").reset_index(drop=True)
        started = perf_counter()
        try:
            predictions, metrics, user_prompt, raw_response = predict_theme_batch(
                gateway,
                theme_id,
                theme_rows,
            )
            call_rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_count": len(theme_rows),
                "status": "ok",
                "latency_seconds": metrics.get("latency_seconds"),
                "input_tokens": metrics.get("input_tokens"),
                "output_tokens": metrics.get("output_tokens"),
                "total_tokens": metrics.get("total_tokens"),
                "error": None,
            })

            for row in theme_rows.to_dict(orient="records"):
                epic_key = str(row["epic_key"]).strip()
                predicted = predictions[epic_key]
                truth = parse_list_value(row["gt_l3_ids"])
                scores = score_sets(predicted, truth)
                result_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_ids": row["stage_ids"],
                    "predicted_l3_ids": predicted,
                    "gt_l3_ids": truth,
                    "status": "ok",
                    "error": None,
                    **scores,
                })
        except Exception as exc:
            call_rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_count": len(theme_rows),
                "status": "error",
                "latency_seconds": perf_counter() - started,
                "input_tokens": None,
                "output_tokens": None,
                "total_tokens": None,
                "error": str(exc),
            })
            for row in theme_rows.to_dict(orient="records"):
                result_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": str(row["epic_key"]).strip(),
                    "stage_ids": row["stage_ids"],
                    "predicted_l3_ids": None,
                    "gt_l3_ids": parse_list_value(row["gt_l3_ids"]),
                    "status": "error",
                    "error": str(exc),
                    "exact_match": None,
                    "precision": None,
                    "recall": None,
                    "f1": None,
                    "predicted_count": None,
                    "truth_count": len(parse_list_value(row["gt_l3_ids"])),
                })

    return pd.DataFrame(result_rows), pd.DataFrame(call_rows)


results, call_metrics = run_experiment()
scored = results.loc[results["status"].eq("ok")].copy()
successful_calls = call_metrics.loc[call_metrics["status"].eq("ok")].copy()

summary = pd.DataFrame([{
    "scope": "fixed_50_valid_epics_seed_42_theme_batch",
    "evaluated_epics": len(scored),
    "exact_match_accuracy": scored["exact_match"].mean() if len(scored) else 0.0,
    "mean_precision": scored["precision"].mean() if len(scored) else 0.0,
    "mean_recall": scored["recall"].mean() if len(scored) else 0.0,
    "mean_f1": scored["f1"].mean() if len(scored) else 0.0,
}])

diagnostics = pd.DataFrame([{
    "sample_seed": SAMPLE_SEED,
    "sample_size": SAMPLE_SIZE,
    "themes_selected": evaluation_population["theme_key"].nunique(),
    "total_epics": len(evaluation_population),
    "successful_calls": int(call_metrics["status"].eq("ok").sum()),
    "failed_calls": int(call_metrics["status"].eq("error").sum()),
    "scored_epics": len(scored),
    "avg_epics_per_call": successful_calls["epic_count"].mean() if len(successful_calls) else 0.0,
}])

latency_tokens = pd.DataFrame([{
    "successful_calls": len(successful_calls),
    "failed_calls": int(call_metrics["status"].eq("error").sum()),
    "avg_latency_seconds": successful_calls["latency_seconds"].mean() if len(successful_calls) else None,
    "p50_latency_seconds": successful_calls["latency_seconds"].quantile(0.50) if len(successful_calls) else None,
    "p95_latency_seconds": successful_calls["latency_seconds"].quantile(0.95) if len(successful_calls) else None,
    "avg_input_tokens": successful_calls["input_tokens"].mean() if len(successful_calls) else None,
    "avg_output_tokens": successful_calls["output_tokens"].mean() if len(successful_calls) else None,
    "avg_total_tokens": successful_calls["total_tokens"].mean() if len(successful_calls) else None,
    "total_input_tokens": successful_calls["input_tokens"].sum() if len(successful_calls) else 0,
    "total_output_tokens": successful_calls["output_tokens"].sum() if len(successful_calls) else 0,
    "total_tokens": successful_calls["total_tokens"].sum() if len(successful_calls) else 0,
    "tokens_per_scored_epic": successful_calls["total_tokens"].sum() / len(scored) if len(scored) else None,
}])

print("Evaluation summary")
display(summary)
print("Population diagnostics")
display(diagnostics)
print("LLM latency / token summary")
display(latency_tokens)
print("Per-call LLM metrics")
display(call_metrics)

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    extra_sheets={
        "evaluation_summary": summary,
        "population_diagnostics": diagnostics,
        "llm_metrics": call_metrics,
        "latency_tokens": latency_tokens,
    },
)
print(f"Saved: {output_path}")
